In [1]:
sample_count = 100
trial = 1

In [2]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [3]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [4]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating this news article. \n <</SYS>> \n\n 
    [INST] Generate a SHORT response of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
    1.If there is one, the city the article is talking about. Otherwise, state that it can't be located. \n 
    2.The specific location within the city you got if you found one. \
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
    If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. This is the article: \n\n
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)


In [5]:
prompt3_1 = PromptTemplate(
    input_variables=["headline", "body"],
    template="""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    
    Cutting Knowledge Date: December 2023
    Today Date: 26 Jul 2024

    You are an expert in geo-location and have a deep understanding of specific places and organizations. In a short response, your task is to identify and provide the most exact real location mentioned in the news article. This can be a place, organization, facility, or any location that can help identify where the article takes place or talks about. Also, mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. Do not discuss anthing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

    You are tasked in geo-locating this news article. Generate a SHORT response specifying the most exact location you can find mentioned in the article. Give your answer in the following format:
    1. If there is one, the city the article is talking about. Otherwise, state that it can't be located. 
    2. The specific place within the city you got if you found one.
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision.
     
    If you cannot determine a location, state that explicitly. DO NOT MAKE UP INFORMATION. This is the news article:
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
)


In [6]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama2_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama3_1_model_path = "./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"


In [7]:
llm2 = LlamaCpp(
    model_path=llama2_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [8]:
llm3_1 = LlamaCpp(
    model_path=llama3_1_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from ./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.license str              = llama3.1
llama_model_l

In [9]:
chain2 = prompt | llm2 | output_parser
chain3_1 = prompt3_1 | llm3_1 | output_parser

In [10]:
# Run LLM on a given article
def run_llm2(headline, body):
    return chain2.invoke({"headline": headline, "body": body})

def run_llm3_1(headline, body):
    return chain3_1.invoke({"headline": headline, "body": body})


## NER Model

In [11]:
import spacy
from span_marker import SpanMarkerModel

In [12]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [13]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [14]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [15]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [16]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [17]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

### Wrapper function to measure time taken by a given function

In [18]:
import time

def sec_to_hms(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = round(seconds % 60)
    return f"{hours:02}:{minutes:02}:{remaining_seconds:02}"

def check_time(func):
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time_formatted = sec_to_hms(total_time)
        print(f"Time taken: {total_time_formatted}")
        return result
    return wrapper

## Pipeline Entry Point

In [19]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one

In [20]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 5 articles
raw_df = full_df.sample(sample_count)
# raw_df = full_df
len(raw_df)


100

In [21]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
3573,00000179-1056-d2db-af7b-755e96c30001,Article,What’s At The Very Bottom Of The Ocean? Rob Mc...,What’s At The Very Bottom Of The Ocean? Rob Mc...,Greater Boston Staff,NaN,Local News,NaN,/local-news/2021/04/26/whats-at-the-very-botto...,Mon Apr 26 22:36:31 EDT 2021,TRUE,Wayland resident Rob McCallum is one of the fe...
11358,00000185-0c81-d270-adb5-5d937a660000,Article,L.L. Bean 897 December 2022 Sweepstakes No Gif...,L.L. Bean 897 December 2022 Sweepstakes No Gif...,00000185-0c81-d270-adb5-5d937aeb0000,NaN,Support,NaN,NaN,Tue Dec 13 12:25:50 EST 2022,TRUE,Enter to win the GBH Radio Contest between 6:0...
2573,00000177-f585-dd2a-a3ff-f5e7c67a0001,Article,Mass. Web Developers Step Up To Offer Vaccine-...,Mass. Web Developers Step Up To Offer Vaccine-...,Greater Boston Staff,NaN,Local News,NaN,/local-news/2021/03/02/mass-web-developers-ste...,Tue Mar 02 20:53:49 EST 2021,TRUE,Following the well-publicized problems with th...
2754,00000178-23c9-db79-a379-7bff7f8f0001,Article,"Biden Aims For Quicker Shots, 'Independence Fr...","Biden Aims For Quicker Shots, 'Independence Fr...",Zeke Miller | Associated Press,NaN,National News,NaN,/national-news/2021/03/11/biden-sets-may-1-tar...,Thu Mar 11 19:23:37 EST 2021,TRUE,WASHINGTON (AP) — One year after the nation wa...
5452,0000017b-9dcc-d950-abfb-bdcc1a7a0001,Article,Hate Crimes Reach The Highest Level In More Th...,Hate Crimes Reach The Highest Level In More Th...,Joe Hernandez,NaN,National News,NaN,/national-news/2021/08/31/hate-crimes-reach-th...,Tue Aug 31 15:17:00 EDT 2021,TRUE,"There were 7,759 reported hate crimes in the U..."
12190,00000186-4a6e-d717-adce-ca7fd21e0002,Article,Reporter's dismissal exposes political pressur...,Reporter's dismissal exposes political pressur...,David Folkenflik,NaN,National News,NaN,/national-news/2023/02/13/reporters-dismissal-...,Mon Feb 13 05:45:00 EST 2023,TRUE,"Late last fall, West Virginia Public Broadcast..."
6540,0000017d-5269-d269-a3fd-fe7b1eef0001,Article,Judge rules to vacate rape conviction in Tyron...,Judge rules to vacate rape conviction in Tyron...,"Sarah Betancourt, Jenifer B. McKim",NaN,Local News,NaN,/local-news/2021/11/24/judge-rules-to-vacate-r...,Wed Nov 24 12:25:15 EST 2021,TRUE,<i>Updated 5:20 p.m. Nov. 24</i><br/><br/>A Su...
7241,0000017e-756d-d440-a17f-77fd40fb0001,Article,Starbucks drops COVID vaccine mandate after Su...,Starbucks drops COVID vaccine mandate after Su...,The Associated Press,NaN,National News,NaN,/national-news/2022/01/20/starbucks-drops-covi...,Wed Jan 19 21:29:00 EST 2022,TRUE,Starbucks is no longer requiring its U.S. work...
851,00000176-44d2-da63-a5fe-e6fb41d00001,Article,This Doctor Was Once An Undocumented Immigrant...,This Doctor Was Once An Undocumented Immigrant...,Greater Boston Staff,NaN,National News,NaN,/national-news/2020/12/08/this-doctor-was-once...,Tue Dec 08 19:33:56 EST 2020,TRUE,"Between late 2018 and 2019, three migrant chil..."
11038,00000184-8575-dd99-ad94-9775deb80001,Article,"On Red Cup Day, thousands of Starbucks workers...","On Red Cup Day, thousands of Starbucks workers...",Vanessa Romo,NaN,National News,NaN,/national-news/2022/11/17/on-red-cup-day-thous...,Thu Nov 17 06:32:00 EST 2022,TRUE,More than a thousand Starbucks employees went ...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [22]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

Remove Duplicates (if any)

In [23]:
duplicates = df.duplicated(subset=['hl1'])

In [24]:
print(duplicates.value_counts())

False    100
Name: count, dtype: int64


In [25]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [26]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

  0%|          | 0/100 [00:00<?, ?it/s]C:\Users\axel0\AppData\Local\Temp\ipykernel_32400\1969457438.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
100%|██████████| 100/100 [00:00<?, ?it/s]


Clean the Body and Header with Regex

In [27]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 100/100 [00:00<?, ?it/s]


### Benchmark total times

In [28]:
time_df = pd.DataFrame(columns=['NER', 'LLM2', 'LLM3.1'])

### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [29]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [30]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    locations_list = []
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                locations_list.append(location)
    
    print(f"\nLocations found on title: \n{locations_list} \n")
    return locations_list

In [31]:
# df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)
df["Explicit_Pass"] = None


In [32]:
df["Explicit_Pass"].value_counts().head(10)

Series([], Name: count, dtype: int64)

### Location Utility Functions

In [33]:
# Normalize a location string
def normalize_location(location):
    location = location.lower().strip() 
    
    # Remove leading terms
    leading_terms = ["a", "an", "the"]

    for term in leading_terms:
        if location.startswith(term + " "):
            location = location[len(term) + 1:]

    # Remove punctuation
    location = re.sub(r'[^\w\s]', '', location)
    
    # Expand common abbreviations
    abbreviation_map = {
        "us": "united states",
        'co': 'company',
        'pd': 'police department',
        'wgbh': 'gbh',
        'cfa': 'the harvard-smithsonian center for astrophysics',
        'pbs': 'public broadcasting service',
        'ap': 'associated press',
        'aim': 'aim (alternative investment market)',
        '&': 'and',
        'npr': 'national public radio',
        'doj': 'the justice department',
        'cdc': 'the centers for disease control and prevention',
        'ssa': 'social security administration',
        'doe': 'the energy department',
        'cbpp': 'the center on budget and policy priorities',
        'mbta': 'massachusetts bay transportation authority',
        't': 'massachusetts bay transportation authority',
        'globe': 'boston globe',
        'senate': 'capitol',
        'congress': 'capitol',
        'legislature': 'capitol',
        'justice': 'department of justice',
        'house': 'u.s. house of representatives',
        'oval office': 'the white house',
        'dese': 'department of elementary and secondary education',
        'bps': 'boston public schools',
        'dhs': 'the department of homeland security',
        'fed': 'federal reserve',
        'fbi': 'the federal bureau of investigation',
        'epa': 'the environmental protection agency',
        'cdc': 'the centers for disease control and prevention',
        'nar': 'national association of realtors',
        'adl': 'anti-defamation league',
        'cbp': 'customs and border protection',
        'cia': 'central intelligence agency',
        'fda': 'food and drug administration',
        'dep': 'department of environmental protection',
        'un': 'united nations',
        'faa': 'federal aviation administration',
        'ntsb': 'national transportation safety board',
        'dua': 'department of unemployment assistance',
        'necn': 'new england cable news',
        'nar': 'national association of realtors',
        'who': 'world health organization',
        'irap': 'international refugee assistance project',
        'ncaa': 'national collegiate athletic association',
        'council': 'city council',
        'dph': 'the department of public health',
        'usda': 'the us department of agriculture',
        'bpr': 'boston public radio',
        'blm': 'black lives matter',
        'irs': 'internal revenue service',
        'necn': 'new england cable news',
        'wh': 'white house',
        'gop': 'the republican party',
        'ps': 'public schools',
        'ma dese': 'the massachusetts department of elementary and secondary education',
    }
    for abbr, full in abbreviation_map.items():
        location = re.sub(r'\b' + abbr + r'\b', full, location)
    
    # TODO: (maybe) Handle synonyms, variants
    # TODO: (maybe) Handle country/state/city abbreviations   
    
    # Remove extra spaces
    location = re.sub(r'\s+', ' ', location)
    
    return location

In [34]:
from fuzzywuzzy import fuzz

# Check if two locations are the same
def are_same_location(loc1, loc2, threshold=80):
    if loc1 == loc2:
        return True
    
    if loc1 in loc2 or loc2 in loc1:
        return True
    
    similarity = fuzz.token_set_ratio(loc1, loc2)
    return similarity >= threshold

# Combine two locations if they are the same
def combine_locations(location, locations):
    if (locations is None or len(locations) == 0):
        return [location]

    # check for news subdomains
    if "gbh" in location:
        location = "gbh"
    elif "npr" in location:
        location = "npr"

    new_locs = []

    for loc in locations:
        if are_same_location(location, loc):
            # Replace the existing location if the new one is longer
            if len(location) > len(loc):
                new_locs.append(location)
            else:
                new_locs.append(loc)
        else:
            new_locs.append(loc)

    new_locs.append(location)

    return new_locs

In [35]:
# Check if a location can be added to the list of locations
def can_add_location(location):
    unwanted_entities = load_cache("./geodata/unwanted_locations.json")

    if location in unwanted_entities:
        return False
    elif invalid_location(location):
        return False
    else:
        return True

# Check if a location is unwanted
def invalid_location(location):
    # List of common unwanted entity types
    unwanted_places = [
        r'\bstreet\b', 
        r'\bsquare\b', 
        r'\bavenue\b', 
        r'\bboulevard\b',
        r'\broad\b', 
        r'\blane\b', 
        r'\bdrive\b', 
        r'\bdriveway\b',
        r'\bhighway\b',
        r'\bfreeway\b'
    ]
    
    # Create a combined regex pattern
    pattern = re.compile('|'.join(unwanted_places))
    
    # Check if the location matches any unwanted entity type
    if pattern.search(location):
        return True
    return False

In [36]:
from collections import Counter

# Get the top 5 most common locations
def get_main_5(facilities, organizations):

    fac_freq = Counter(facilities)
    org_freq = Counter(organizations)

    top_fac = fac_freq.most_common(1) if facilities else []
    top_org = org_freq.most_common(1) if organizations else []

    combined = facilities + organizations
    combined_freq = Counter(combined)

    if top_fac:
        combined_freq.pop(top_fac[0][0], None)
    if top_org:
        combined_freq.pop(top_org[0][0], None)
    
    top_combined = combined_freq.most_common(3)

    top_entities = top_fac + top_org + top_combined
    
    top_entities = [entity[0] for entity in top_entities]

    return top_entities

In [37]:
all_locations_path = "./geodata/all_locations.json"
all_locations = load_cache(all_locations_path)

In [38]:
def add_entity(entity, valid_list):
    loc = normalize_location(entity)
    if (can_add_location(loc)):
        valid_list = combine_locations(loc, valid_list)
        all_locations.append(loc) # For testing
    
    return valid_list


In [39]:
abbreviation_path = "./geodata/abbreviations.json"
abbreviations = load_cache(abbreviation_path)

In [40]:
# Return all valid facilities and organizations found
@check_time
def get_valid_entities(entities):
    valid_facs = []
    valid_orgs = []

    if (entities is None or len(entities) == 0):
        return None
    
    # For Testing

    abbreviations = load_cache(abbreviation_path)

    def is_abbreviation(text):
        return len(text) < 6 and text.isupper()

    new_entities = []
    for entity in entities:
        if (entity.label_ == "FAC" or entity.label_ == "ORG"):
            location = entity.text
            if is_abbreviation(location):
                if location not in abbreviations:
                    abbreviations.append(location)

            new_entities.append(location)
    print("All valid entities: ", new_entities)
    save_cache_to_file(abbreviations, abbreviation_path)
    # End Testing

    for entity in entities:
        if (entity.label_ == "FAC"):
            valid_facs = add_entity(entity.text, valid_facs)
        elif (entity.label_ == "ORG"):
            valid_orgs = add_entity(entity.text, valid_orgs)

    
    valid_entities = get_main_5(valid_facs, valid_orgs)
    
    if (len(valid_entities) == 0):
        return None
    else:
        return valid_entities
        

### NER Code First Pass

In [41]:
# Run NER on the body of the article and return first valid facility
@check_time
def run_NER(text):
    if (text == None or text == ""):
        return None
    
    try:
        entities = nlp(text).ents
        valid_entities = get_valid_entities(entities)
        print(f"\nMain valid entities found: {valid_entities} \n")
        return valid_entities
        
    except Exception as error:
        print(error)
        return None

In [42]:
# Chunk processing - split the article into chunks of text and run NER on each chunk
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size, chunk_limit=None):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    all_entities = []

    if chunk_limit is not None:
        if len(chunks) > chunk_limit:
            chunks = chunks[:chunk_limit]
    
    # Process each chunk and return if valid entities are found
    for chunk in chunks:
        result = run_NER(chunk)
        if result is not None:
             all_entities.extend(result)
    
    print(f"\nAll entities for the article: \n{all_entities} \n")
    
    if len(all_entities) == 0:
        return None
    else:
        return all_entities

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [43]:
@check_time
def handle_chunk_processing(article):
    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text, chunk_limit=5)
    except Exception as error:
        print(error)

In [44]:
start_time = time.time()

df["NER_Pass"] = df["body"].progress_apply(run_NER)
# df["NER_Pass"] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Time taken: {total_time_formatted}")
time_df.loc[0, "NER"] = total_time_formatted


  2%|▏         | 2/100 [00:04<03:55,  2.41s/it]

All valid entities:  []
Time taken: 00:00:00

Main valid entities found: None 

Time taken: 00:00:05


  3%|▎         | 3/100 [00:10<06:03,  3.75s/it]

All valid entities:  []
Time taken: 00:00:00

Main valid entities found: None 

Time taken: 00:00:06


  4%|▍         | 4/100 [00:21<10:04,  6.30s/it]

...

All valid entities:  ['the GBH Drama Club', 'The First Churchills', 'MASTERPIECE', 'BBC', 'BBC', 'Tony', 'Royal Shakespeare Company', 'MI5', 'the London Metropolitan Police', 'HBO', 'Buckingham Palace']
Time taken: 00:00:00

Main valid entities found: [('buckingham palace', 1), ('bbc', 2), ('gbh drama club', 1), ('first churchills', 1), ('masterpiece', 1)] 

Time taken: 00:04:17


100%|██████████| 100/100 [3:27:13<00:00, 174.61s/it]

All valid entities:  ['the Boston Symphony Orchestra', 'GBH', 'BSO', 'the Berlin Wall']
Time taken: 00:00:00

Main valid entities found: [('berlin wall', 1), ('boston symphony orchestra', 1), ('gbh', 1), ('bso', 1)] 

Time taken: 00:03:45


100%|██████████| 100/100 [3:28:33<00:00, 125.14s/it]

All valid entities:  ['GBH', 'the deCordova Sculpture Park and Museum', 'SpeakEasy Stage Company', 'Netflix', 'Netflix', 'Camp Jened', 'Jened', 'the Disability Rights Movement']
Time taken: 00:00:00

Main valid entities found: [('decordova sculpture park and museum', 1), ('netflix', 2), ('camp jened', 1), ('gbh', 1), ('speakeasy stage company', 1)] 

Time taken: 00:01:20
Time taken: 03:28:34


In [45]:
save_cache_to_file(all_locations, all_locations_path)
print(all_locations)

['gbh', '897 gbh', 'gbh', 'gbh jazz', 'jazz 24', 'massachusetts bay transportation authority', 'massachusetts bay transportation authority', 'blue line', 'bowdoin station', 'charles mgh', 'red line', 'bowdoin station', 'blue line', 'charles mgh', 'massachusetts general hospital', 'red and blue lines', 'massachusetts bay transportation authority', 'blue line', 'fiscal and management control board', 'loews hotels', 'capitol', 'twitter', 'loews hotels', 'fighting for missouri', 'loews', 'universal orlando', 'electoral college', 'congress', 'capitol', 'simon schuster', 'kansas city star', 'st louis post dispatch', 'star', 'capitol', 'lincoln project', 'the republican party', 'united states u.s. senate', 'national public radio', 'souza baranowski correctional center', 'lemuel shattuck hospital', 'tottenham', 'gbh news', 'tottenham', 'gbh news center for investigative reporting', 'gbh news', 'gbh news', 'tottenham', 'shattuck', 'prisoners legal services', 'carrillo', 'gbh news', 'gbh news ce

In [46]:
abbreviations = load_cache(abbreviation_path)
print(abbreviations)

['WGBH', 'MBTA', 'GOP', 'NPR', 'GBH', 'ESPN', 'CNN', 'BPR', 'FBI', 'AP', 'EPA', 'NECN', 'SHNS', 'NAR', 'WHO', 'ADL', 'PBS', 'GE', 'CBP', 'IRAP', 'DOT', 'CDC', 'FDA', 'J&J', 'ADP', 'MIT', 'NCAA', 'DEP', 'U.N.', 'GBH 2', 'COVID', 'FAA', 'NTSB', 'NORAD', 'U2', 'BPS', 'EU', 'NATO', 'NFL', 'WNBA', 'NBC', 'DUA', 'IMANE', 'FIPA', 'MLB', 'GLAD', 'IRS', 'SCDAO', 'FTX', 'GM', 'M&M', 'BBC', 'MI5', 'HBO', 'BSO']


In [47]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass
3573,00000179-1056-d2db-af7b-755e96c30001,What At The Very Bottom Of The Ocean Rob McCal...,Wayland resident Rob McCallum is one of the fe...,None,None
11358,00000185-0c81-d270-adb5-5d937a660000,L.L. Bean 897 December 2022 Sweepstakes No Gif...,Enter to win the GBH Radio Contest between 00A...,None,None
2573,00000177-f585-dd2a-a3ff-f5e7c67a0001,Mass. Web Developers Step Up To Offer Vaccine ...,Following the well publicized problems with th...,None,"[(twitter, 1)]"
2754,00000178-23c9-db79-a379-7bff7f8f0001,Biden Aims For Quicker Shots Independence From...,WASHINGTON (AP) One year after the nation was ...,None,"[(east room, 1), (white house, 3), (oval offic..."
5452,0000017b-9dcc-d950-abfb-bdcc1a7a0001,Hate Crimes Reach The Highest Level In More Th...,There were 759 reported hate crimes in the U.S...,None,"[(fbi, 6), (anti defamation league, 1), (adl, ..."
12190,00000186-4a6e-d717-adce-ca7fd21e0002,Reporter dismissal exposes political pressures...,Late last fall West Virginia Public Broadcasti...,None,"[(capitol, 2), (west virginia educational broa..."
6540,0000017d-5269-d269-a3fd-fe7b1eef0001,Judge rules to vacate rape conviction in Tyron...,Updated 20 p.m. Nov. 24A Suffolk County Superi...,None,"[(gbh news, 2), (suffolk county superior court..."
7241,0000017e-756d-d440-a17f-77fd40fb0001,Starbucks drops COVID vaccine mandate after Su...,Starbucks is no longer requiring its U.S. work...,None,"[(starbucks, 6), (carhartt, 3), (supreme court..."
851,00000176-44d2-da63-a5fe-e6fb41d00001,This Doctor Was Once An Undocumented Immigrant...,Between late 2018 and 2019 three migrant child...,None,"[(united states customs and border protection,..."
11038,00000184-8575-dd99-ad94-9775deb80001,On Red Cup Day thousands of Starbucks workers ...,More than thousand Starbucks employees went on...,None,"[(starbucks workers united, 9), (national publ..."


### Llama Prediction

In [48]:
def filter_llama_output(log):
    # Define regex patterns to match the lines we want to remove
    llama_print_timings_pattern = re.compile(r'llama_print_timings:.*')
    llama_generate_pattern = re.compile(r'Llama.generate:.*')

    lines = log.split('\n')

    filtered_lines = []

    for line in lines:
        # If the line matches any of the unwanted patterns, skip it
        if llama_print_timings_pattern.match(line) or llama_generate_pattern.match(line):
            continue

        filtered_lines.append(line.strip())

    # Join the filtered lines back into a single string
    filtered_log = '\n'.join(filtered_lines)
    
    return filtered_log

In [49]:
# Run LLM model on the articles, and then run the NER on the prediction.
@check_time
def predict_llama2(article):
    try:
        truncated_text = article['body'][:4000]
        llama_prediction = run_llm2(article['hl1'], truncated_text)
        cleaned_prediction = filter_llama_output(llama_prediction)
        print(f"\nLlama 2 Prediction: \n{cleaned_prediction} \n")
        
        valid_entities = run_NER(cleaned_prediction)
        print(f"\nAll entities for the article from LLM 2: \n{valid_entities} \n")
        return valid_entities
    
    except Exception as error:
        print(error)
        return None

In [50]:
start_time = time.time()

# df['LLM_2_Pass'] = df.progress_apply(predict_llama2, axis=1)
df['LLM_2_Pass'] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM2"] = total_time_formatted

Total time taken: 00:00:00


In [51]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass
3573,00000179-1056-d2db-af7b-755e96c30001,What At The Very Bottom Of The Ocean Rob McCal...,Wayland resident Rob McCallum is one of the fe...,None,None,None
11358,00000185-0c81-d270-adb5-5d937a660000,L.L. Bean 897 December 2022 Sweepstakes No Gif...,Enter to win the GBH Radio Contest between 00A...,None,None,None
2573,00000177-f585-dd2a-a3ff-f5e7c67a0001,Mass. Web Developers Step Up To Offer Vaccine ...,Following the well publicized problems with th...,None,"[(twitter, 1)]",None
2754,00000178-23c9-db79-a379-7bff7f8f0001,Biden Aims For Quicker Shots Independence From...,WASHINGTON (AP) One year after the nation was ...,None,"[(east room, 1), (white house, 3), (oval offic...",None
5452,0000017b-9dcc-d950-abfb-bdcc1a7a0001,Hate Crimes Reach The Highest Level In More Th...,There were 759 reported hate crimes in the U.S...,None,"[(fbi, 6), (anti defamation league, 1), (adl, ...",None
12190,00000186-4a6e-d717-adce-ca7fd21e0002,Reporter dismissal exposes political pressures...,Late last fall West Virginia Public Broadcasti...,None,"[(capitol, 2), (west virginia educational broa...",None
6540,0000017d-5269-d269-a3fd-fe7b1eef0001,Judge rules to vacate rape conviction in Tyron...,Updated 20 p.m. Nov. 24A Suffolk County Superi...,None,"[(gbh news, 2), (suffolk county superior court...",None
7241,0000017e-756d-d440-a17f-77fd40fb0001,Starbucks drops COVID vaccine mandate after Su...,Starbucks is no longer requiring its U.S. work...,None,"[(starbucks, 6), (carhartt, 3), (supreme court...",None
851,00000176-44d2-da63-a5fe-e6fb41d00001,This Doctor Was Once An Undocumented Immigrant...,Between late 2018 and 2019 three migrant child...,None,"[(united states customs and border protection,...",None
11038,00000184-8575-dd99-ad94-9775deb80001,On Red Cup Day thousands of Starbucks workers ...,More than thousand Starbucks employees went on...,None,"[(starbucks workers united, 9), (national publ...",None


In [52]:
@check_time
def predict_llama3_1(article):
    try:
        truncated_text = article['body'][:4000]
        llama_prediction = run_llm3_1(article['hl1'], truncated_text)
        cleaned_prediction = filter_llama_output(llama_prediction)
        print(f"\nLlama 3.1 Prediction: \n{cleaned_prediction} \n")

        valid_entities = run_NER(cleaned_prediction)
        print(f"\nAll entities for the article from LLM 3.1: \n{valid_entities} \n")
        return valid_entities
    
    except Exception as error:
        print(error)
        return None

In [53]:
start_time = time.time()

# df['LLM_3_1_Pass'] = df.progress_apply(predict_llama3_1, axis=1)
df["LLM_3_1_Pass"] = None
end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM3.1"] = total_time_formatted

Total time taken: 00:00:00


In [54]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass
3573,00000179-1056-d2db-af7b-755e96c30001,What At The Very Bottom Of The Ocean Rob McCal...,Wayland resident Rob McCallum is one of the fe...,None,None,None,None
11358,00000185-0c81-d270-adb5-5d937a660000,L.L. Bean 897 December 2022 Sweepstakes No Gif...,Enter to win the GBH Radio Contest between 00A...,None,None,None,None
2573,00000177-f585-dd2a-a3ff-f5e7c67a0001,Mass. Web Developers Step Up To Offer Vaccine ...,Following the well publicized problems with th...,None,"[(twitter, 1)]",None,None
2754,00000178-23c9-db79-a379-7bff7f8f0001,Biden Aims For Quicker Shots Independence From...,WASHINGTON (AP) One year after the nation was ...,None,"[(east room, 1), (white house, 3), (oval offic...",None,None
5452,0000017b-9dcc-d950-abfb-bdcc1a7a0001,Hate Crimes Reach The Highest Level In More Th...,There were 759 reported hate crimes in the U.S...,None,"[(fbi, 6), (anti defamation league, 1), (adl, ...",None,None
12190,00000186-4a6e-d717-adce-ca7fd21e0002,Reporter dismissal exposes political pressures...,Late last fall West Virginia Public Broadcasti...,None,"[(capitol, 2), (west virginia educational broa...",None,None
6540,0000017d-5269-d269-a3fd-fe7b1eef0001,Judge rules to vacate rape conviction in Tyron...,Updated 20 p.m. Nov. 24A Suffolk County Superi...,None,"[(gbh news, 2), (suffolk county superior court...",None,None
7241,0000017e-756d-d440-a17f-77fd40fb0001,Starbucks drops COVID vaccine mandate after Su...,Starbucks is no longer requiring its U.S. work...,None,"[(starbucks, 6), (carhartt, 3), (supreme court...",None,None
851,00000176-44d2-da63-a5fe-e6fb41d00001,This Doctor Was Once An Undocumented Immigrant...,Between late 2018 and 2019 three migrant child...,None,"[(united states customs and border protection,...",None,None
11038,00000184-8575-dd99-ad94-9775deb80001,On Red Cup Day thousands of Starbucks workers ...,More than thousand Starbucks employees went on...,None,"[(starbucks workers united, 9), (national publ...",None,None


In [55]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

0


Series([], Name: count, dtype: int64)

In [56]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

96


NER_Pass
[(twitter, 1)]                                                                                                                            1
[(east room, 1), (white house, 3), (oval office, 1), (u.s. senate, 1), (rice university, 1)]                                              1
[(dua, 10), (lawyers for civil rights, 2), (massachusetts department of unemployment assistance, 1), (harvard legal aid bureau, 1)]       1
[(boston public radio, 6), (biden administration, 2), (banda rim bam bum, 2), (made by google, 1)]                                        1
[(covid tracking project, 1)]                                                                                                             1
                                                                                                                                         ..
[(gbh drama, 1)]                                                                                                                          1
[(patriots,

In [57]:
print(df['LLM_2_Pass'].value_counts().sum())
df['LLM_2_Pass'].value_counts()

0


Series([], Name: count, dtype: int64)

In [58]:
print(df['LLM_3_1_Pass'].value_counts().sum())
df['LLM_3_1_Pass'].value_counts()

0


Series([], Name: count, dtype: int64)

In [59]:
len(df)

100

In [60]:
df.to_csv(f"./results/benchmarking_{sample_count}_samples_trial_{trial}.csv")
time_df.to_csv(f"./results/benchmarking_times_{sample_count}_samples_trial_{trial}.csv")

In [61]:
print(df.count())
print(df.dropna(subset=['NER_Pass'])["LLM_2_Pass"].notnull().sum())
print(df.dropna(subset=['NER_Pass'])["LLM_3_1_Pass"].notnull().sum())

print(df.dropna(subset=['LLM_2_Pass'])["LLM_3_1_Pass"].notnull().sum())

_id              100
hl1              100
body             100
Explicit_Pass      0
NER_Pass          96
LLM_2_Pass         0
LLM_3_1_Pass       0
dtype: int64
0
0
0


Extract locations from the most specific pass

In [62]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'LLM_2_Pass', 'LLM_3_1_Pass']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [63]:
# df['Locations'] = df.progress_apply(extractLocations, axis=1)

In [64]:
from fuzzywuzzy import fuzz

def normalize_location(location):
    location = location.lower().strip()
    if location.startswith("the "):
        location = location[4:]
    return location

def are_same_location(loc1, loc2, threshold=85):
    norm_loc1 = normalize_location(loc1)
    norm_loc2 = normalize_location(loc2)
    similarity = fuzz.token_set_ratio(norm_loc1, norm_loc2)
    return similarity >= threshold

def extractAllLocations(article):
    locations_list = []
    for key in ['Explicit_Pass', 'NER_Pass', 'LLM_2_Pass', 'LLM_3_1_Pass']:
        location = article.get(key)
        if location is not None:
            locations_list.extend(location)
    
    if len(locations_list) == 0:
        return None
    else:
        # Ensure unique locations considering variations
        unique_locations = []
        for loc in locations_list:
            if not any(are_same_location(loc, unique_loc) for unique_loc in unique_locations):
                unique_locations.append(loc)
        return unique_locations

In [65]:
df['Locations'] = df.progress_apply(extractAllLocations, axis=1)

  4%|▍         | 4/100 [00:00<?, ?it/s]


AttributeError: 'tuple' object has no attribute 'lower'

In [ ]:
df.head(10)

## Get the coordinates

In [ ]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [ ]:
# Get the coordinates of the location
def getCoordinates(location): 
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [ ]:
def getAllCoordinates(locations):
    coordinates_list = []

    if (locations == None): return None
    
    for location in locations:
        coordinates = getCoordinates(location)
        if coordinates is not None:
            coordinates_list.append(coordinates)
    return coordinates_list

In [ ]:
df['Coordinates'] = df['Locations'].progress_apply(getAllCoordinates)

In [ ]:
df.head(10)

## Geocode locations

In [ ]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [ ]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County

In [ ]:
def getAllGeocodes(locations):
    tracts = []
    counties = []

    if (locations == None): return None, None
    
    for location in locations:
        Tract, County = geocode(location)
        tracts.append(Tract)
        counties.append(County)

    return tracts, counties

In [ ]:
df[['Tracts', 'Counties']] = df['Locations'].progress_apply(getAllGeocodes).apply(pd.Series)


In [ ]:
df.head(10)

## Get Neighborhoods

In [ ]:
original_neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800"
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
	

}

In [ ]:
neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800", 
    "365100", "361300"
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
	

}

In [ ]:
def string_to_list(s):
    if(s != ''):
        return [s]
    else:
        return []  # Return the string as a single-element list

In [ ]:
def find_neighborhood_by_tract(search_dict, tract_to_find):
	for key, values in search_dict.items():
		if (tract_to_find in values):
			return key
	return "Unknown Neighborhood"

def locateNeighborhoods(tract):
	if (tract == None):
		return None

	query = find_neighborhood_by_tract(neigh_tract_dict, tract)
	if (query != None):
		return string_to_list(query)
	else: 
		return string_to_list("None")


In [ ]:
unknown_tracts_path = "./geodata/unknown_tracts.json"
unknown_tracts = load_cache(unknown_tracts_path)

In [ ]:
def handle_tract_to_neighborhood(articles):
    tracts = articles['Tracts']

    neighborhoods = []

    if (tracts == None): return None

    for i, tract in enumerate(tracts):
        neighborhood = locateNeighborhoods(tract)
        if neighborhood is not None:
            if (neighborhood[0] == "Unknown Neighborhood"):
                location = articles['Locations'][i]
                if (tract not in unknown_tracts):
                    unknown_tracts[tract] = {"County": articles['Counties'][i], "Locations": [location]}
                    print(f"Unknown neighborhood for location: {location} with tract: {tract}")
                elif (location not in unknown_tracts[tract]["Locations"]):
                    unknown_tracts[tract]["Locations"].append(location)
                    print(f"Unknown neighborhood for location: {location} with tract: {tract}")
            neighborhoods.extend(neighborhood)

    save_cache_to_file(unknown_tracts, unknown_tracts_path)
    return neighborhoods

In [ ]:
df["Neighborhoods"] = df.progress_apply(handle_tract_to_neighborhood, axis=1)

In [ ]:
df.head(10)

In [ ]:
def remove_unknown_neighborhoods(df):
    def remove_unknowns(row):
        neighborhoods = row['Neighborhoods']
        if neighborhoods is None:
            return row
        
        indexes_to_remove = [i for i, neighborhood in enumerate(neighborhoods) if neighborhood == 'Unknown Neighborhood']
        
        for column in ['Tracts', 'Counties', 'Coordinates', 'Neighborhoods']:
            if row[column] is not None:
                row[column] = [v for i, v in enumerate(row[column]) if i not in indexes_to_remove]
        # row['Locations'] = [v for i, v in enumerate(row['Locations']) if i not in indexes_to_remove]
        
        return row

    df = df.apply(remove_unknowns, axis=1)
    return df

In [ ]:
df = remove_unknown_neighborhoods(df)

In [ ]:
df

In [ ]:
df.to_csv(f"./results/full_benchmark_{sample_count}_samples_trial_{trial}.csv")
time_df.to_csv(f"./results/full_benchmark_times_{sample_count}_samples_trial_{trial}.csv")